## Create Gas Flow

In [ ]:
import numpy as np
import cantera as ct
gas = ct.Solution('ch4_smooke_EKT.yaml')

Starting conditions

In [ ]:
cases_CH4 = [0, 0.05, 0.05]
particle_mass = 5e-9 #equivalent to particle mass release rate

rhos = []
has = []
fuels = { 
        'fuel1': {'CH2O': 0.4, 'CH3O': 0.4, 'CH3': 0.2}, 
        'fuel2': {'CH4': 1}, 
        'fuel3': {'CO': 0.3474, 'N2': 0.6526} 
        }

for CH4 in cases_CH4:
    Yair = ( 1 - CH4)
    Y = {'O2': 0.233 * Yair, 'N2': 0.767 * Yair, 'CH4': CH4}
    gas.TPY = 400, ct.one_atm, Y
    rhos.append(gas.density)
    has.append(gas.enthalpy_mass)

print(rhos) 
print(has)

In [ ]:
inlet_area = 1e-3**2
inlet_area

In [ ]:
rate_fuel1 = np.array([2,2,1.6])
rate_CO = np.array([0,0,1.4])
rate_O2 = np.array([0,0,-0.8])

mass_fractions_fuel1 = 0.5
# These are necessary for OpenFOAM
mass_fractions_O2 = 0.25
mass_fractions_CO = 0.25

particle_mass = 5e-9
m_particle_fuel1 = particle_mass * rate_fuel1 * mass_fractions_fuel1
m_particle_fuel3 = particle_mass * rate_CO * mass_fractions_CO
m_particle_O2 = particle_mass * rate_O2 * mass_fractions_O2

m_particle = m_particle_fuel1 + m_particle_fuel3 + m_particle_O2
m_particle

In [ ]:
# multiply times 9 so that the volatiles are exactly 10 % 
m_flow = np.ones(3) * 9 * m_particle[0]
m_flow

In [ ]:
Us = m_flow / rhos / inlet_area
Us

In [ ]:
print('has',has)
print('Us',Us)

## Define case settings

In [ ]:
import pandas as pd
df = pd.DataFrame({
        'HA':has,
        'UZ':Us,
        'FUELS': ['Sfuel1 fuel1', 'Sfuel1 fuel1 fuel2', 'Sfuel1 fuel1 fuel2 Sfuel3 fuel3'],
        'Y_FUEL2':[0,0.05,0.05],
        'SUMMANDS':['fuel1','fuel1 fuel2','fuel1 fuel2 fuel3'],
        'Ys': ['','Y','Y YY'], 
        'NUMYY':['','','fuel3'],
        'DENYY':['','','fuel2 fuel3'],
        'NUMY':['','fuel2','fuel2 fuel3'],
        'DENY':['','fuel1 fuel2','fuel1 fuel2 fuel3'],
        'RATE_FUEL1':rate_fuel1,
        'RATE_FUEL3_O2':rate_O2,
        'RATE_FUEL3_CO':rate_CO
        })
df.to_csv('OF_settings.csv')

## Reference

The reference can be calculated analytically.

In [ ]:
import pandas as pd
reference = {'massflow_inlet':[],'massflow_outlet':[],'T':[],'fuel1':[],'fuel2':[],'fuel3':[],'CH4':[],'O2':[],'CO':[],'N2':[],'CH3O':[]}

for i in range(3):
    reference['massflow_inlet'].append(m_flow[i])
    reference['massflow_outlet'].append(m_flow[i]+m_particle[i])
    # outlet values
    reference['T'].append(400)
    reference['fuel1'].append(m_particle_fuel1[i] / (m_particle[i] + m_flow[i]))
    reference['fuel2'].append(cases_CH4[i] * m_flow[i] / (m_particle[i] + m_flow[i]))
    reference['fuel3'].append(m_particle_fuel3[i] * 2.879 / (m_particle[i] + m_flow[i])) 
    reference['CH4'].append(cases_CH4[i] * m_flow[i] / (m_particle[i] + m_flow[i]))
    W_C, W_O = 12, 16
    reference['O2'].append(((1-cases_CH4[i])*0.233 * m_flow[i] - m_particle_fuel3[i]*(16/(12+16))) / (m_particle[i] + m_flow[i]))
    reference['CO'].append(m_particle_fuel3[i]/(m_flow[i]+m_particle[i]))
    reference['N2'].append((1-cases_CH4[i])*0.767*m_flow[i]/(m_flow[i]+m_particle[i]))
    reference['CH3O'].append(m_particle_fuel1[i]*0.4/(m_flow[i]+m_particle_fuel1[i]))

reference_df = pd.DataFrame(reference)
reference_df

In [ ]:
reference_df.to_csv('reference/data/reference_data.csv')

## Create plots

In [ ]:
import pyvista as pv
import numpy as np

Read OF case

In [ ]:
import os
vtks = []
for case_path in ['test_volatile','test_volatile_plus_gas','test_volatile_plus_gas_plus_char_off']:
    reader = pv.POpenFOAMReader(os.path.join('temp',case_path,'case.foam'))
    reader.case_type = 'reconstructed'
    reader.set_active_time_value(0.3)
    vtks.append(reader.read())


In [ ]:
lines = []
for vtk in vtks:
    internal = vtk['internalMesh']
    x, y = 0.e-3, 0.e-3
    if 'fuel2' not in internal.array_names:
        internal['fuel2'] = np.zeros(internal.n_cells)
    if 'fuel3' not in internal.array_names:
        internal['fuel3'] = np.zeros(internal.n_cells)
    lines.append(internal.sample_over_line((x,y,0),(x,y,np.max(internal.points[:,2])),resolution=100))
# J -> kJ
for line in lines:
    line['ha'] = line['ha']/1000

Profiles

In [ ]:
from matplotlib import pyplot as plt

fig, axs = plt.subplots(2,4)

fig.set_size_inches(20,10)
for ax, name, unit  in zip(axs.ravel(), ['fuel1','fuel2','fuel3','ha','O2','N2','CO','T'],['(-)','(-)','(-)','(kJ/kg)','(-)','(-)','(-)','(K)']):
    for line in lines:
        ax.plot(line.points[:,2]*1000,line[name])
        ax.set_xlabel('$z$ (mm)')
        ax.set_ylabel(name+' '+unit)
    ax.grid()
    if not ax.get_subplotspec().is_last_row():  # Hide x labels if not in last row
        ax.set_xlabel("")
        ax.set_xticklabels([])
    if name == 'T':
        ax.set_ylim(398,402)
legend_lines = []
import matplotlib.lines as mlines
for i,color in enumerate(['C0','C1','C2']):
    legend_lines.append(mlines.Line2D([], [], color=color, linestyle="-", label="Case"+str(i)))
fig.subplots_adjust(wspace=0.3)
fig.legend(handles=legend_lines, ncol=3, fontsize=14, loc="lower center" ,bbox_to_anchor=(0.5, -0.05))

In [ ]:
fig.savefig('reference/figures/plots.png',bbox_inches='tight')

In [ ]:
pd.read_csv('report/report2.csv',index_col=0)